# Pitch detection demo (offline)

Notebook demonstruje porównanie estymacji wysokości dźwięku przy użyciu `librosa` (YIN) oraz wskazówek dotyczących użycia CREPE.

Instalacja (PowerShell):
```
pip install librosa soundfile matplotlib numpy pandas
# opcjonalnie: pip install crepe
```

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

%matplotlib inline

## Wczytanie pliku audio i normalizacja

In [ ]:
audio_path = 'sample.wav'  # zamień na ścieżkę do pliku
if not Path(audio_path).exists():
    print('Brak pliku sample.wav w katalogu. Wstaw własne nagranie.')
y, sr = librosa.load(audio_path, sr=44100)
# normalizacja RMS
rms = np.sqrt(np.mean(y**2))
if rms > 0:
    y = y / (rms + 1e-9)
print(f'Ładowanie {audio_path}, sr={sr}, długość={len(y)/sr:.2f}s')

## Estymacja F0 z użyciem librosa.yin (podobne do pYIN)

In [ ]:
fmin = librosa.note_to_hz('C2')
fmax = librosa.note_to_hz('C7')
f0_yin = librosa.yin(y, fmin=fmin, fmax=fmax, sr=sr, frame_length=2048, hop_length=256)
times = librosa.frames_to_time(np.arange(len(f0_yin)), sr=sr, hop_length=256)
plt.figure(figsize=(10,3))
plt.plot(times, f0_yin, label='YIN')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Estimated F0 (librosa.yin)')
plt.legend()
plt.show()

## CREPE (opcjonalnie)
Jeśli masz zainstalowany `crepe`, możesz użyć modelu pre-trained do estymacji F0. Poniżej przykład wywołania (komentarz).

In [ ]:
# Przykład (jeśli crepe zainstalowane):
# import crepe
# f0_crepe, confidence, activation = crepe.predict(audio_path, sr, step_size=10, viterbi=True)
# times_crepe = np.arange(len(f0_crepe)) * 0.01
# plt.plot(times_crepe, f0_crepe, label='CREPE')
# plt.legend()
# plt.show()
print('Sekcja CREPE: zainstaluj crepe, odkomentuj kod i uruchom')

## Porównanie i zapis wyników

In [ ]:
# Przy braku referencji wygenerujemy prosty reference przez wygładzenie f0_yin
f0_ref = pd.Series(f0_yin).rolling(window=5, min_periods=1, center=True).median().to_numpy()
# Oblicz RMSE (ignorujemy NaN)
mask = ~np.isnan(f0_ref) & ~np.isnan(f0_yin)
rmse_yin = np.sqrt(np.mean((f0_ref[mask] - f0_yin[mask])**2))
print(f'RMSE (YIN vs smoothed ref) = {rmse_yin:.2f} Hz')
# Zapis do CSV
out_df = pd.DataFrame({'time': times, 'f0_yin': f0_yin, 'f0_ref': f0_ref})
out_df.to_csv('pitch_results.csv', index=False)
print('Wyniki zapisane do pitch_results.csv')

## Notatki
- Dostosuj `frame_length` i `hop_length` do rozdzielczości czasowej i częstotliwościowej wymogów.
- CREPE działa z `step_size` w ms (np. 10 ms).
- W eksperymentach porównawczych używaj tej samej reprezentacji czasowej (synchronizacja ramek) dla wszystkich metod.